In [1]:
import os
from sentence_transformers import SentenceTransformer

ModuleNotFoundError: No module named 'sentence_transformers'

In [ ]:
# set the Groq API key via environment variable
groq_api_key = os.environ.get('GROQ_API_KEY')
if not groq_api_key:
    raise EnvironmentError(
        'GROQ_API_KEY is not set. Please set it before running this notebook. '
        'Example PowerShell: $env:GROQ_API_KEY="<your-key>"; Example Bash: export GROQ_API_KEY="<your-key>"'
    )

# also expose it for Hugging Face access if the model loader needs it
os.environ['HUGGINGFACEHUB_API_TOKEN'] = groq_api_key

OSError: GROQ_API_KEY is not set. Please set it before running this notebook. Example PowerShell: $env:GROQ_API_KEY="<your-key>"; Example Bash: export GROQ_API_KEY="<your-key>"

In [ ]:
#step 1 load document
from langchain_community.document_loaders import TextLoader
loader=TextLoader('rajini_dataset.txt')
documents=loader.load()

In [ ]:
#step 2 create the embedding + vector DB
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={"device": "cpu"})
vector_db = FAISS.from_documents(documents, embeddings)


In [ ]:
#step 3 Retrieval + Generation
def rag_query(query):
  docs = vector_db.similarity_search(query, k=3)
  context = " ".join(doc.page_content for doc in docs)
  prompt = f"Context: {context}\n\nQuestion: {query}\nAnswer:"
  return generate_response(prompt)

In [ ]:
# Get the query from environment for non-interactive runs, otherwise use a default
import os
query_to_test = os.environ.get('NOTEBOOK_QUERY', "Please summarize the hospital policy in one paragraph.")
print(f"Using query: {query_to_test}")

In [ ]:
#step 3: dataset preparation
from datasets import Dataset
with open('hospital_data.txt', 'r', encoding='utf-8') as f:
    text = f.read().strip()
data = [{"text": text}]
dataset = Dataset.from_list(data)

In [ ]:
#step 4: tokenization
import os
if os.environ.get('SKIP_TRAINING') == '1':
    print('SKIP_TRAINING=1, skipping tokenization and downstream training steps.')
    tokenized_dataset = None
else:
    def tokenize_function(example):
      return tokenizer(example["text"],truncation=True,padding="max_length")
    tokenized_dataset=dataset.map(tokenize_function)

In [ ]:
import os
if os.environ.get('SKIP_TRAINING') == '1':
  print('SKIP_TRAINING=1, skipping training step.')
else:
  #step 5: Training
  from transformers import Trainer , TrainingArguments
  #add labels to the tokenized_dataset for causal language mdelling
  def add_labels_to_dataset(examples):
    examples['labels']=examples['input_ids']
    return examples
  tokenized_dataset=tokenized_dataset.map(add_labels_to_dataset,batched=True)
  training_args=TrainingArguments(
    output_dir="./lora_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=10,
    save_steps=50
  )
  trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
  )
  trainer.train()

In [ ]:
import os
if os.environ.get('SKIP_TRAINING') == '1':
    def generate_response(prompt):
        return "[Skipping generation in automated run: SKIP_TRAINING=1]"
else:
    #step 6: inferences
    def generate_response(prompt):
      inputs=tokenizer(prompt,return_tensors="pt").to("cuda")
      outputs=model.generate(**inputs,max_new_tokens=100)
      return tokenizer.decode(outputs[0])

In [ ]:
print(generate_response("what is ROI"))

In [ ]:
print(rag_query(query_to_test))